# 01 · Entendimento do negócio e dos dados (v3)
**CRISP-DM:** Entendimento do Negócio + Entendimento dos Dados

## 1. Entendimento do negócio
**Problema.** Quando um incidente é aberto, o operador precisa saber se as APIs (recursos) envolvidas estão operando dentro de um **regime normal** e, se não estão, **qual indicador saiu do padrão**: erros de negócio (4xx), erros de sistema (5xx) ou latência. O comportamento normal de uma API muda com o horário, o dia e os processamentos em lote, e por isso não basta um limite fixo.

**Decisão apoiada.** *O recurso estava em regime normal no momento do registro do incidente? Se não, qual indicador foi impactado e desde quando?* Como informação complementar: o impacto é **generalizado** ou concentrado em um **consumidor específico** capaz de alterar o resultado agregado?

**Objetivo de mineração.** Para cada recurso: (1) caracterizar seus regimes normais a partir dos Access Logs agregados; (2) no instante do registro de um incidente, classificar o período imediatamente anterior como **Normal** ou **Residual**; (3) apontar o **indicador responsável** e estimar o **início** do desvio.

**Questões de pesquisa.**
- **RQ1:** Access Logs agregados permitem caracterizar os regimes normais de cada recurso e decidir, no instante do registro, se o recurso está em regime normal?
- **RQ2:** Quando o recurso não está normal, o método identifica corretamente o indicador responsável (4xx, 5xx, latência) e o início do desvio?
- **RQ3:** Qual o menor período antes do registro (15, 30 ou 60 min) que já permite essa decisão, e como o atraso entre a degradação e o registro afeta o resultado?

**Critérios de sucesso** (definidos antes da modelagem):

| Critério | Meta |
|---|---|
| Consultas de controle (sem falha) com saída Normal | ≥ 95% |
| Sensibilidade no registro, faixa de 15 min, intensidade média ou alta | ≥ 70% |
| Acurácia do indicador responsável (casos detectados) | ≥ 90% |
| Início real dentro da zona estimada | ≥ 70% |
| Sensibilidade do método proposto ≥ baselines (McNemar) | sem perda significativa |

In [1]:
%run ./00_funcoes.ipynb

00_funcoes (v10) carregado · /home/wsl/gh/mestrado/claude9/v10


## 2. Entendimento dos dados
**Cada linha da base é um agregado por minuto × recurso × código HTTP × cliente**: `QT_TTL_TRAN` é o total de execuções naquele minuto e `QT_TMP_MEDI_EXEA`, o tempo médio de execução delas. Não há registros por requisição; por isso, percentis de latência são aproximados a partir dessas médias por minuto (notebook 02). `TS_EVT` está em UTC e é convertido para o horário de Brasília. `CD_CLI` vem anonimizado (SHA-256 com semente).

In [2]:
bruto = carregar()
print(f"linhas: {len(bruto):,}")
print("período (Brasília):", bruto["ts"].min(), "→", bruto["ts"].max())
print("recursos:", bruto["recurso"].nunique(), "· clientes:", bruto["cliente"].nunique())
print("status:", sorted(bruto["status"].unique()))
print("nulos:", int(bruto.isna().sum().sum()), "· qtd ≤ 0:", int((bruto["qtd"] <= 0).sum()), "· tmp < 0:", int((bruto["tmp"] < 0).sum()))

linhas: 5,602,126
período (Brasília): 2026-05-01 00:00:00 → 2026-09-19 23:59:00
recursos: 12 · clientes: 1976
status: [np.int64(0), np.int64(200), np.int64(201), np.int64(400), np.int64(401), np.int64(403), np.int64(412), np.int64(416), np.int64(422), np.int64(423), np.int64(429), np.int64(499), np.int64(500), np.int64(502), np.int64(503), np.int64(504)]
nulos: 0 · qtd ≤ 0: 0 · tmp < 0: 0


### Códigos dos recursos
No artigo, os recursos são identificados por **E01–E12**. A v3 **reaproveita o mapeamento da v2** (ordem de volume em agosto) para manter a correspondência entre as versões; o mapeamento fica só nos resultados.

In [3]:
mapa = codigos()
assert set(mapa) == set(bruto["recurso"].unique()), "recursos diferentes da v2"
pd.Series(mapa, name="código")

18169917    E01
18917906    E02
18917920    E03
18344302    E04
18166630    E05
18917921    E06
18170306    E07
18139260    E08
18168196    E09
17834257    E10
18917905    E11
14553746    E12
Name: código, dtype: str

### Comparação com a base anterior (v1/v2)
No período comum (01/08–08/09), a base nova tem exatamente o mesmo volume da anterior, mas os hashes de cliente são diferentes (outra semente de anonimização). Os clientes não podem ser cruzados entre as duas bases; dentro da base nova, o hash é consistente.

In [4]:
ant = PROJETO / "DADOS" / "APIs-Agr-20260801-20260908_hash.csv"
if ant.exists():
    o = carregar(ant)
    m = (bruto["ts"] >= o["ts"].min()) & (bruto["ts"] <= o["ts"].max())
    print("volume no período comum: nova", int(bruto.loc[m, "qtd"].sum()), "· anterior", int(o["qtd"].sum()))
    print("clientes da base anterior presentes na nova:", f"{o['cliente'].isin(set(bruto['cliente'])).mean():.1%}")

volume no período comum: nova 35888265 · anterior 35888265


clientes da base anterior presentes na nova: 0.0%


### Volume médio diário e taxa de 4xx por mês
Mudanças de patamar entre meses (crescimento de volume, mudança da taxa de erros de negócio) motivam a comparação de histórico de 3 meses × 1 mês.

In [5]:
mes = bruto["ts"].dt.to_period("M")
vol_dia = bruto.groupby(["recurso", mes])["qtd"].sum() / bruto.groupby(["recurso", mes])["ts"].apply(lambda s: s.dt.date.nunique())
vol_dia = (vol_dia.unstack() / 1000).round(1)
vol_dia.index = vol_dia.index.map(rot)
t4 = bruto[bruto["status"] // 100 == 4].groupby(["recurso", mes])["qtd"].sum() / bruto.groupby(["recurso", mes])["qtd"].sum()
t4 = t4.unstack().round(3)
t4.index = t4.index.map(rot)
display(vol_dia.sort_index())
display(t4.sort_index())
vol_dia.to_csv(RESULTADOS / "01_volume_diario_mes.csv")
t4.to_csv(RESULTADOS / "01_taxa4xx_mes.csv")

ts,2026-05,2026-06,2026-07,2026-08,2026-09
recurso,,,,,
E01,275.7,309.4,337.6,322.9,372.2
E02,NaN,NaN,273.1,217.1,150.2
E03,NaN,NaN,60.2,61.3,49.4
E04,27.4,37.2,39.1,42.9,46.0
E05,29.3,39.3,48.9,43.7,43.7
E06,NaN,NaN,38.5,51.7,31.9
E07,32.7,34.8,39.4,40.0,51.0
E08,31.9,39.8,43.7,39.9,47.9
E09,36.1,41.1,39.5,37.5,34.7


ts,2026-05,2026-06,2026-07,2026-08,2026-09
recurso,,,,,
E01,0.017,0.016,0.014,0.006,0.014
E02,NaN,NaN,0.322,0.451,0.622
E03,NaN,NaN,0.364,0.337,0.458
E04,0.005,0.044,0.062,0.128,0.113
E05,0.024,0.009,0.032,0.008,0.001
E06,NaN,NaN,0.781,0.798,0.771
E07,0.115,0.064,0.037,0.032,0.025
E08,0.127,0.095,0.086,0.110,0.065
E09,0.017,0.014,0.010,0.007,0.007


### Chaves repetidas

In [6]:
chave = ["ts", "recurso", "status", "cliente"]
rep = bruto.duplicated(chave, keep=False)
qualidade = dict(
    linhas=len(bruto), linhas_em_chave_repetida=int(rep.sum()), linhas_excedentes=int(bruto.duplicated(chave).sum()),
    frac_volume_em_chave_repetida=float(bruto.loc[rep, "qtd"].sum() / bruto["qtd"].sum()),
    iguais_nas_metricas=int(bruto[rep].duplicated(chave + ["qtd", "tmp"]).sum()))
qualidade

{'linhas': 5602126,
 'linhas_em_chave_repetida': 52506,
 'linhas_excedentes': 35026,
 'frac_volume_em_chave_repetida': 0.008130351238269693,
 'iguais_nas_metricas': 24832}

### Cobertura por recurso

In [7]:
cob = bruto.groupby("recurso").agg(inicio=("ts", "min"), fim=("ts", "max"), minutos=("ts", "nunique"),
                                   clientes=("cliente", "nunique"), volume=("qtd", "sum"))
cob["cobertura_min"] = cob["minutos"] / (((cob["fim"] - cob["inicio"]) / pd.Timedelta("1min")) + 1)
cob["part_volume"] = cob["volume"] / cob["volume"].sum()
cob["transacoes_por_15min_mediana"] = bruto.assign(j=bruto["ts"].dt.floor("15min")).groupby(["recurso", "j"])["qtd"].sum().groupby("recurso").median()
cob.index = cob.index.map(mapa)
cob = cob.sort_index()
cob

,inicio,fim,minutos,clientes,volume,cobertura_min,part_volume,transacoes_por_15min_mediana
recurso,,,,,,,,
E01,2026-05-01 00:00:00,2026-09-19 23:59:00,203636,38,45378806,0.995872,0.472166,1212.0
E02,2026-07-16 14:05:00,2026-09-03 07:11:00,69676,555,11550710,0.993286,0.120185,2239.0
E03,2026-07-16 14:04:00,2026-09-03 07:11:00,67777,246,3012404,0.966200,0.031344,498.0
E04,2026-05-25 18:57:00,2026-09-19 23:58:00,132195,24,4723317,0.783229,0.049146,149.0
E05,2026-05-01 00:00:00,2026-09-19 23:59:00,194377,19,5790385,0.950592,0.060249,113.0
E06,2026-07-16 14:05:00,2026-09-03 07:11:00,68085,787,2313818,0.970605,0.024075,290.0
E07,2026-05-01 00:00:00,2026-09-19 23:59:00,166506,36,5488004,0.814290,0.057103,123.0
E08,2026-05-01 00:00:00,2026-09-19 23:59:00,203521,39,5685922,0.995310,0.059162,279.0
E09,2026-05-01 00:01:00,2026-09-19 23:59:00,161699,30,5396661,0.790785,0.056152,132.0


### Proporção por família HTTP e latência

In [8]:
fam = bruto.groupby(bruto["status"] // 100)["qtd"].sum()
display((fam / fam.sum()).round(4))
desc = bruto[["qtd", "tmp"]].describe(percentiles=[0.5, 0.9, 0.99]).T
extremos = dict(latencia_acima_60s=int((bruto["tmp"] > 60_000).sum()), latencia_zero=int((bruto["tmp"] == 0).sum()))
display(desc)
extremos

status
0    0.0000
2    0.8762
4    0.1217
5    0.0021
Name: qtd, dtype: float64

,count,mean,std,min,50%,90%,99%,max
qtd,5602126.0,17.155590,82.884504,1.0,3.0,25.0,267.0,4711.0
tmp,5602126.0,84.480103,1899.139504,0.0,42.0,86.0,418.0,1015962.0


{'latencia_acima_60s': 92, 'latencia_zero': 3686}

## 3. Regras de validade
1. Conversão de `TS_EVT` de UTC para America/Sao_Paulo antes de qualquer corte temporal.
2. Consolidação das chaves repetidas (soma de volume, média de latência ponderada).
3. Latências extremas mantidas (estatísticas robustas).
4. Cada recurso é avaliado apenas enquanto há dados: cinco recursos começam em 16/07, um em 25/05, e quatro param em 03/09.
5. Uma única linha com código HTTP 0 é mantida (entra no volume, não nas taxas de 4xx/5xx).

In [9]:
qualidade.update(extremos, periodo_inicio=bruto["ts"].min(), periodo_fim=bruto["ts"].max(),
                 recursos=int(bruto["recurso"].nunique()), clientes=int(bruto["cliente"].nunique()),
                 familias={int(k): float(v) for k, v in (fam / fam.sum()).items()})
salvar_json(qualidade, "01_qualidade.json")
cob.to_csv(RESULTADOS / "01_cobertura_recursos.csv")